# 第二十四课:训练循环——真训《西游记》

讲义:[24-train-loop.md](24-train-loop.md)。第 20/23 课的 Mini-GPT + 反向传播,接上第 13 课 AdamW,在《西游记》全本上真训 2000 步(约 2 分钟)。

In [ ]:
import math
import time
import numpy as np

# ===== 第 20 课 MiniGPT 原码(含 generate) =====
def softmax(x, axis=-1):
    m = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=axis, keepdims=True)

class MiniGPT:
    def __init__(self, vocab=16, d=32, n_heads=4, n_layers=2, seed=0):
        rng = np.random.default_rng(seed)
        s = 0.1
        self.E = rng.normal(size=(vocab, d)) * 0.5
        self.dh = d // n_heads
        self.H, self.D, self.NL = n_heads, d, n_layers
        self.layers = [{
            'Wq': rng.normal(size=(d, d))*s, 'Wk': rng.normal(size=(d, d))*s,
            'Wv': rng.normal(size=(d, d))*s, 'Wo': rng.normal(size=(d, d))*s,
            'W1': rng.normal(size=(d, 4*d))*s*0.8, 'W2': rng.normal(size=(4*d, d))*s*0.8,
            'W3': rng.normal(size=(d, 4*d))*s*0.8,
        } for _ in range(n_layers)]
        self.head = rng.normal(size=(d, vocab)) * s

    def rms(self, x, eps=1e-5):
        return x / np.sqrt((x**2).mean(-1, keepdims=True) + eps)

    def rope(self, x, pos):
        half = x.shape[-1] // 2
        ang = pos[:, None] / (10000 ** (np.arange(half)*2/x.shape[-1]))
        c, s = np.cos(ang), np.sin(ang)
        x1, x2 = x[..., :half], x[..., half:]
        return np.concatenate([x1*c - x2*s, x1*s + x2*c], -1)

    def _attn(self, x, pos, W, cache):
        n = x.shape[0]
        q, k, v = self.rope(x@W['Wq'], pos), self.rope(x@W['Wk'], pos), x@W['Wv']
        if cache is not None:
            k, v = np.concatenate([cache[0], k]), np.concatenate([cache[1], v])
        T = k.shape[0]
        mask = np.triu(np.ones((n, T), bool), T - n + 1)
        outs = []
        for h in range(self.H):
            sl = slice(h*self.dh, (h+1)*self.dh)
            s = q[:, sl] @ k[:, sl].T / math.sqrt(self.dh)
            s = np.where(mask, -np.inf, s)
            outs.append(softmax(s) @ v[:, sl])
        return np.concatenate(outs, -1) @ W['Wo'], (k, v)

    def _ffn(self, x, W):
        h = x @ W['W1']
        return (h / (1 + np.exp(-h)) * (x @ W['W3'])) @ W['W2']

    def forward(self, ids, cache=None):
        pos0 = cache[0][0].shape[0] if cache else 0
        x = self.E[list(ids)]
        new_cache = []
        for i, W in enumerate(self.layers):
            a, kv = self._attn(x, np.arange(pos0, pos0+len(ids)), W,
                               cache[i] if cache else None)
            x = x + a
            x = x + self._ffn(self.rms(x), W)
            new_cache.append(kv)
        logits = self.rms(x) @ self.head
        assert np.isfinite(logits).all(), 'NaN! 查掩码/精度/ε'
        return logits, new_cache

    def generate(self, ids, n, T=1.0, top_p=1.0, rng=None):
        rng = rng or np.random.default_rng()
        logits, cache = self.forward(ids)
        out = []
        for _ in range(n):
            p = softmax(logits[-1] / T)
            if top_p < 1.0:
                order = np.argsort(-p); cum = np.cumsum(p[order])
                keep = order[:np.searchsorted(cum, top_p)+1]
                m = np.zeros_like(p); m[keep] = p[keep]; p = m/m.sum()
            nxt = int(rng.choice(len(p), p=p/p.sum()))
            out.append(nxt)
            logits, cache = self.forward([nxt], cache)
        return list(ids) + out

In [ ]:
# ===== 第 23 课:反向四件套 + loss_and_grads + 扁平化/裁剪 =====
def rms_backward(dy, x, eps=1e-5):
    r = np.sqrt((x**2).mean(-1, keepdims=True) + eps)
    y = x / r
    return (dy - y * (dy * y).mean(-1, keepdims=True)) / r

def rope_backward(dy, pos, dim):
    half = dim // 2
    ang = pos[:, None] / (10000 ** (np.arange(half)*2/dim))
    c, s = np.cos(ang), np.sin(ang)
    dy1, dy2 = dy[..., :half], dy[..., half:]
    return np.concatenate([dy1*c + dy2*s, -dy1*s + dy2*c], -1)

def silu_grad(u):
    sig = 1 / (1 + np.exp(-u))
    return sig * (1 + u * (1 - sig))

def ce_loss(logits, targets):
    z = logits - logits.max(-1, keepdims=True)
    logp = z - np.log(np.exp(z).sum(-1, keepdims=True))
    return -logp[np.arange(len(targets)), targets].mean()

def loss_and_grads(gpt, ids, targets):
    T = len(ids)
    pos = np.arange(T)
    tape = []
    x = gpt.E[list(ids)]
    for W in gpt.layers:
        q, k, v = gpt.rope(x@W['Wq'], pos), gpt.rope(x@W['Wk'], pos), x@W['Wv']
        mask = np.triu(np.ones((T, T), bool), 1)
        ps, outs = [], []
        for h in range(gpt.H):
            sl = slice(h*gpt.dh, (h+1)*gpt.dh)
            s = q[:, sl] @ k[:, sl].T / math.sqrt(gpt.dh)
            s = np.where(mask, -np.inf, s)
            p = softmax(s)
            ps.append(p); outs.append(p @ v[:, sl])
        concat_o = np.concatenate(outs, -1)
        x_mid = x + concat_o @ W['Wo']
        x_rms = gpt.rms(x_mid)
        u, zv = x_rms @ W['W1'], x_rms @ W['W3']
        gate = u / (1 + np.exp(-u)) * zv
        x_out = x_mid + gate @ W['W2']
        tape.append(dict(x_in=x, x_mid=x_mid, x_rms=x_rms, u=u, zv=zv,
                         gate=gate, concat_o=concat_o, q=q, k=k, v=v, ps=ps))
        x = x_out

    x_final = x
    logits = gpt.rms(x_final) @ gpt.head
    loss = ce_loss(logits, targets)

    grads = {'E': np.zeros_like(gpt.E), 'head': np.zeros_like(gpt.head),
             'layers': [dict.fromkeys(W) for W in gpt.layers]}
    p = softmax(logits)
    onehot = np.zeros_like(p)
    onehot[np.arange(T), targets] = 1.0
    dlogits = (p - onehot) / T
    grads['head'] = gpt.rms(x_final).T @ dlogits
    dx = rms_backward(dlogits @ gpt.head.T, x_final)

    for i in reversed(range(gpt.NL)):
        W, t = gpt.layers[i], tape[i]
        dg = dx @ W['W2'].T
        du = dg * silu_grad(t['u']) * t['zv']
        dzv = dg * (t['u'] / (1 + np.exp(-t['u'])))
        grads['layers'][i]['W2'] = t['gate'].T @ dx
        grads['layers'][i]['W1'] = t['x_rms'].T @ du
        grads['layers'][i]['W3'] = t['x_rms'].T @ dzv
        dx_mid = dx + rms_backward(du @ W['W1'].T + dzv @ W['W3'].T, t['x_mid'])
        dconcat = dx_mid @ W['Wo'].T
        grads['layers'][i]['Wo'] = t['concat_o'].T @ dx_mid
        dvs, dqs, dks = [], [], []
        for h in range(gpt.H):
            sl = slice(h*gpt.dh, (h+1)*gpt.dh)
            p_h, v_h, q_h, k_h = t['ps'][h], t['v'][:, sl], t['q'][:, sl], t['k'][:, sl]
            dO = dconcat[:, sl]
            dvs.append(p_h.T @ dO)
            dp = dO @ v_h.T
            ds = p_h * (dp - (dp * p_h).sum(-1, keepdims=True))
            dqs.append(ds @ k_h / math.sqrt(gpt.dh))
            dks.append(ds.T @ q_h / math.sqrt(gpt.dh))
        dv = np.concatenate(dvs, -1)
        dq = rope_backward(np.concatenate(dqs, -1), pos, gpt.D)
        dk = rope_backward(np.concatenate(dks, -1), pos, gpt.D)
        grads['layers'][i]['Wv'] = t['x_in'].T @ dv
        grads['layers'][i]['Wq'] = t['x_in'].T @ dq
        grads['layers'][i]['Wk'] = t['x_in'].T @ dk
        dx = dx_mid + dq @ W['Wq'].T + dk @ W['Wk'].T + dv @ W['Wv'].T

    np.add.at(grads['E'], np.array(ids), dx)
    return loss, grads

def flat_params(gpt):
    ps = [gpt.E, gpt.head]
    for W in gpt.layers:
        ps += [W[k] for k in ('Wq', 'Wk', 'Wv', 'Wo', 'W1', 'W2', 'W3')]
    return ps

def flat_grads(grads):
    gs = [grads['E'], grads['head']]
    for L in grads['layers']:
        gs += [L[k] for k in ('Wq', 'Wk', 'Wv', 'Wo', 'W1', 'W2', 'W3')]
    return gs

def clip_norms(gs, max_norm=1.0):
    total = math.sqrt(sum((g ** 2).sum() for g in gs))
    if total > max_norm:
        scale = max_norm / total
        for g in gs:
            g *= scale
    return total

In [ ]:
# ===== 数据管道(讲义 §2) =====
text = open("data/xiyouji.txt", encoding="utf-8").read()
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = np.array([stoi[c] for c in text], dtype=np.int32)
n_train = int(len(data) * 0.99)
train_data, val_data = data[:n_train], data[n_train:]
V = len(chars)
print(f"语料 {len(data):,} 字 | 词表 V={V} | 训练 {len(train_data):,} / 验证 {len(val_data):,}")
print(f"瞎猜基线 ln({V}) = {math.log(V):.4f}")

def get_batch(split, B, T, rng):
    src = train_data if split == 'train' else val_data
    off = rng.integers(0, len(src) - T - 1, B)
    x = np.stack([src[o:o + T] for o in off])
    y = np.stack([src[o + 1:o + T + 1] for o in off])
    return x, y

In [ ]:
# ===== 优化器 + 学习率 + batch 梯度(讲义 §3–4) =====
class MultiAdamW:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, wd=0.01):
        self.ps, self.lr, self.b1, self.b2, self.eps, self.wd = params, lr, *betas, eps, wd
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(self, gs):
        self.t += 1
        for p, g, m, v in zip(self.ps, gs, self.m, self.v):
            m *= self.b1; m += (1 - self.b1) * g
            v *= self.b2; v += (1 - self.b2) * g * g
            m_hat = m / (1 - self.b1 ** self.t)
            v_hat = v / (1 - self.b2 ** self.t)
            p -= self.lr * (m_hat / (np.sqrt(v_hat) + self.eps) + self.wd * p)

def lr_at(step, total, base_lr=1e-3, warmup=100, floor_ratio=0.1):
    if step < warmup:
        return base_lr * (step + 1) / warmup
    p = (step - warmup) / max(1, total - warmup)
    return base_lr * (floor_ratio + (1 - floor_ratio) * 0.5 * (1 + math.cos(math.pi * p)))

def batch_grads(gpt, x, y):
    B, T = x.shape
    gsum, tot = None, 0.0
    for b in range(B):
        loss, grads = loss_and_grads(gpt, list(x[b]), list(y[b]))
        tot += loss
        gs = flat_grads(grads)
        if gsum is None:
            gsum = [g / B for g in gs]
        else:
            for acc, g in zip(gsum, gs):
                acc += g / B
    return tot / B, gsum

In [ ]:
# ===== 训练循环(讲义 §4–5,完整 2000 步,约 2 分钟) =====
def sample_text(gpt, prompt, n=60, T_temp=0.8, seed=0):
    ids = [stoi[c] for c in prompt]
    out = gpt.generate(ids, n, T=T_temp, top_p=0.95, rng=np.random.default_rng(seed))
    return ''.join(itos[i] for i in out)

gpt = MiniGPT(vocab=V, d=64, n_heads=4, n_layers=2, seed=0)
params = flat_params(gpt)
opt = MultiAdamW(params, lr=1e-3, wd=0.01)
rng = np.random.default_rng(42)
STEPS, B, T = 2000, 16, 64

val_x, val_y = get_batch('val', B, T, np.random.default_rng(7))
l0, _ = batch_grads(gpt, val_x, val_y)
print(f"未训练 loss = {l0:.4f}(基线 {math.log(V):.4f} 以上为正常,第 20 课解释过为什么)")

SNAP_PROMPT = "那猴王"
snapshots = {0: sample_text(gpt, SNAP_PROMPT, 48, 0.8, seed=1)}

print(f"\n配置:d=64 4头 2层 {sum(p.size for p in params):,} 参数 | batch {B}×T{T} | {STEPS} 步")
print(f"{'step':>5} {'train':>8} {'val':>8} {'gnorm':>7} {'lr':>9}")
t_start = time.time()
for step in range(STEPS):
    opt.lr = lr_at(step, STEPS)
    x, y = get_batch('train', B, T, rng)
    loss, gs = batch_grads(gpt, x, y)
    gn = clip_norms(gs, 1.0)
    opt.step(gs)
    if (step + 1) % 200 == 0 or step == 0:
        vl, _ = batch_grads(gpt, val_x, val_y)
        print(f"{step+1:>5} {loss:>8.4f} {vl:>8.4f} {gn:>7.3f} {opt.lr:>9.2e}")
    if (step + 1) in (500, 1000, 2000):
        snapshots[step + 1] = sample_text(gpt, SNAP_PROMPT, 48, 0.8, seed=1)

print(f"\n训练用时 {(time.time()-t_start)/60:.1f} min;"
      f"tokens 见过 {STEPS*B*T:,} ≈ {STEPS*B*T/len(train_data):.1f} 个 epoch")

In [ ]:
print("--- 训练中途的生成快照(「那猴王」,T=0.8,top_p=0.95,同种子)---")
for k in sorted(snapshots):
    print(f"[step {k:>4}] {snapshots[k]}")

print("\n换个提示:")
print(sample_text(gpt, "行者道：", 60, T_temp=0.8, seed=1))

vl, _ = batch_grads(gpt, val_x, val_y)
print(f"\n最终 val loss = {vl:.4f} = {vl/math.log(2):.2f} bits/char"
      f"(基线 {math.log(V):.4f};压到基线的 {vl/math.log(V):.0%})")

In [ ]:
# ---- 作业现场 ----

# TODO(C1): 关讲义默写 get_batch,断言 x.shape == y.shape == (B, T)

# TODO(C2): 打印 lr_at(step, 2000) 在 0..2000 的曲线,确认热身/退火形状

# TODO(D2): 同配置训 6000 步,记录 val 曲线找过拟合起点(注意运行时间 ~6 分钟)

# TODO(E1): 把 STEPS 改 6000 重跑本 notebook,对照讲义 §5 的曲线解读

# TODO(E2/预习): 问模型「一加一等于几?」——直接用 base 模型续写看看会发生什么
# print(sample_text(gpt, "一加一等于", 30, T_temp=0.5, seed=3))
